# Setup

Pull in respective libraries to prepare the notebook environment.

In [1]:
!git clone https://github.com/ultralytics/yolov5  # clone
%cd yolov5
%pip install -qr requirements.txt  # install

import torch
import utils
display = utils.notebook_init()  # checks

YOLOv5 🚀 v7.0-368-gb163ff8d Python-3.10.12 torch-2.4.1+cu121 CUDA:0 (Tesla T4, 15102MiB)


Setup complete ✅ (2 CPUs, 12.7 GB RAM, 36.3/112.6 GB disk)


In [2]:
# Ensure we're in the right directory to download our custom dataset
import os
os.makedirs("../datasets/", exist_ok=True)
%cd ../datasets/

/content/datasets


In [3]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="YOUR_API_KEY")
project = rf.workspace("YOUR_WORKSPACE").project("YOUR_PROJECT")
version = project.version(1)
dataset = version.download("folder")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.3/80.3 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 3.8 MB/s eta 0:00:00
  Attempting uninstall: idna
    Found existing installation: idna 3.10
    Uninstalling idna-3.10:
      Successfully uninstalled idna-3.10
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to BrainTumour-1 in folder:: 100%|██████████| 5028/5028 [00:01<00:00, 3482.26it/s]


In [4]:
#Save the dataset name to the environment so we can use it in a system call later
dataset_name = dataset.location.split(os.sep)[-1]
os.environ["DATASET_NAME"] = dataset_name

### Train On Custom Data 🎉
Here, we use the DATASET_NAME environment variable to pass our dataset to the `--data` parameter.

Note: we're training for 100 epochs here. We're also starting training from the pretrained weights. Larger datasets will likely benefit from longer training.

In [5]:
%cd ../yolov5
from utils.downloads import attempt_download

p5 = ['n', 's', 'm', 'l', 'x']  # P5 models
cls = [f'{x}-cls' for x in p5]  # classification models

for x in cls:
    attempt_download(f'weights/yolov5{x}.pt')

/content/yolov5


100%|██████████| 4.87M/4.87M [00:00<00:00, 78.0MB/s]

100%|██████████| 10.5M/10.5M [00:00<00:00, 84.5MB/s]

100%|██████████| 24.9M/24.9M [00:00<00:00, 100MB/s]

100%|██████████| 50.9M/50.9M [00:00<00:00, 123MB/s]

100%|██████████| 92.0M/92.0M [00:00<00:00, 117MB/s]



In [6]:
%cd ../yolov5
!python classify/train.py --model efficientnet_b3.pt --data $DATASET_NAME --epochs 50 --img 640 --cache --batch-size 16 --verbose

/content/yolov5
2024-09-26 10:26:46.128536: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-09-26 10:26:46.156235: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-09-26 10:26:46.164895: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
classify/train: model=efficientnet_b3.pt, data=BrainTumour-1, epochs=50, batch_size=16, imgsz=640, nosave=False, cache=ram, device=, workers=8, project=runs/train-cls, name=exp, exist_ok=False, pretrained=True, optimizer=Adam, lr0=0.001, decay=5e-05, label_smoothing=0.1, cutoff=None, dropout=None, verbose=True, seed=0, local_rank=-1
github: up to 

### Validate Your Custom Model

Repeat step 2 from above to test and validate your custom model.

In [7]:
!python classify/val.py --weights runs/train-cls/exp/weights/best.pt --data ../datasets/$DATASET_NAME --verbose --img-size 640

classify/val: data=../datasets/BrainTumour-1, weights=['runs/train-cls/exp/weights/best.pt'], batch_size=128, imgsz=640, device=, workers=8, verbose=True, project=runs/val-cls, name=exp, exist_ok=False, half=False, dnn=False
YOLOv5 🚀 v7.0-368-gb163ff8d Python-3.10.12 torch-2.4.1+cu121 CUDA:0 (Tesla T4, 15102MiB)

testing:   0% 0/4 [00:00<?, ?it/s]/content/yolov5/classify/val.py:111: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=device.type != "cpu"):
testing: 100% 4/4 [00:14<00:00,  3.56s/it]
                   Class      Images    top1_acc    top5_acc
                     all         501       0.988           1
            glioma_tumor         153        0.98           1
        meningioma_tumor         123       0.976           1
                no_tumor          63           1           1
         pituitary_tumor         162           1           1
Speed: 0.4ms pre-pro

### Infer With Your Custom Model

In [9]:
#Get the path of an image from the test or validation set
if os.path.exists(os.path.join(dataset.location, "test")):
  split_path = os.path.join(dataset.location, "test")
else:
  os.path.join(dataset.location, "valid")
example_class = os.listdir(split_path)[0]
example_image_name = os.listdir(os.path.join(split_path, example_class))[0]
example_image_path = os.path.join(split_path, example_class, example_image_name)
os.environ["TEST_IMAGE_PATH"] = example_image_path

print(f"Inferring on an example of the class '{example_class}'")

#Infer
!python classify/predict.py --weights runs/train-cls/exp/weights/best.pt --source $TEST_IMAGE_PATH

Inferring on an example of the class 'glioma_tumor'
classify/predict: weights=['runs/train-cls/exp/weights/best.pt'], source=/content/datasets/BrainTumour-1/test/glioma_tumor/gg-521-_jpg.rf.d3c868080e7006275609a90f91cb1be5.jpg, data=data/coco128.yaml, imgsz=[224, 224], device=, view_img=False, save_txt=False, nosave=False, augment=False, visualize=False, update=False, project=runs/predict-cls, name=exp, exist_ok=False, half=False, dnn=False, vid_stride=1
YOLOv5 🚀 v7.0-368-gb163ff8d Python-3.10.12 torch-2.4.1+cu121 CUDA:0 (Tesla T4, 15102MiB)

image 1/1 /content/datasets/BrainTumour-1/test/glioma_tumor/gg-521-_jpg.rf.d3c868080e7006275609a90f91cb1be5.jpg: 224x224 no_tumor 0.41, glioma_tumor 0.37, pituitary_tumor 0.17, meningioma_tumor 0.05, 13.8ms
Speed: 0.4ms pre-process, 13.8ms inference, 55.3ms NMS per image at shape (1, 3, 224, 224)
Results saved to runs/predict-cls/exp
